# symbopt tutorial

This executable tutorial starts from the compact public API and then opens the hood: planning, certification, semialg delegation, SOS/SDP proposals, critical cones, verified intervals, and structural internals.

## 1. Five-minute quickstart

In [ ]:
import sympy as sp
from symbopt import (
    SDPSolverConfig, analyze_critical_cone, available_external_sdp_solvers,
    build_moment_matrix, build_sparsity_graph, enclose_expression, minimize,
    minimize_result, plan_problem, semialg_available, sos_decompose,
)

x, y = sp.symbols("x y", real=True)
quickstart_result = minimize((x - 2)**2, variables=x)
assert quickstart_result == [0, {x: 2}]
quickstart_result


## 2. Constraints, bounds, variables, And/Or

In [ ]:
constraint_forms = {
    "single": minimize(x**2, x >= 1, variables=x),
    "and": minimize(x**2, sp.And(x >= 1, x <= 3), variables=x),
    "iterable": minimize(x**2, [x >= 1, x <= 3], variables=x),
    "bounds": minimize(x**2, variables=x, bounds={x: (1, 3)}),
}
assert len(set(map(repr, constraint_forms.values()))) == 1
disjunction_result = minimize(x**2, sp.Or(x <= -1, x >= 1), variables=x)
constraint_forms, disjunction_result


## 3. Compact results vs detailed results

In [ ]:
detailed_result = minimize_result(x**4 - 2*x**2, variables=x)
assert detailed_result.certified
assert detailed_result.optimum_value == -1
print(detailed_result.method)
print(detailed_result.explain())


## 4. How method="auto" chooses a solver

In [ ]:
planner_example = plan_problem(
    x**4 + x*y + y**4,
    [x**2 + y**2 <= 2],
    [x, y],
    {x: (-2, 2), y: (-2, 2)},
)
print(planner_example.complexity)
print(planner_example.reasons)
print(planner_example.explain())
assert planner_example.complexity.polynomial
assert planner_example.complexity.finite_box


## 5. Exact vs numerical vs certified

In [ ]:
proof_demo = minimize_result((x - sp.Rational(1, 3))**2, variables=x)
assert proof_demo.certified and proof_demo.attained
assert proof_demo.optimum_value == 0
print(proof_demo.explain())


## 6. LP/QP examples

In [ ]:
qp_demo = minimize_result(
    (x - 1)**2 + (y - 2)**2,
    [x >= 0, y >= 0, x + y <= 4],
    variables=[x, y],
)
assert qp_demo.certified
assert qp_demo.optimum_value == 0
print(qp_demo.method, qp_demo.explain())


## 7. Polynomial global optimization

In [ ]:
sos_polynomial = (x + y)**2 + (x - y)**2
polynomial_demo = minimize_result(sos_polynomial, variables=[x, y])
assert polynomial_demo.optimum_value == 0
assert polynomial_demo.certified
polynomial_demo.explain()


## 8. semialg exact optimization

In [ ]:
semialg_demo = {"available": semialg_available()}
if semialg_demo["available"]:
    semialg_result = minimize_result(
        x**2 + y**2,
        [x + y >= 1],
        variables=[x, y],
        semialg_certification="complete",
    )
    semialg_demo.update(value=semialg_result.optimum_value, certified=semialg_result.certified)
semialg_demo


## 9. SOS and external SDP backends

In [ ]:
solvers = available_external_sdp_solvers()
if "CLARABEL" in solvers:
    sdp_config = SDPSolverConfig(backend="cvxpy", external_solver="CLARABEL", tol=1e-8)
elif "SCS" in solvers:
    sdp_config = SDPSolverConfig(backend="cvxpy", external_solver="SCS", tol=1e-8)
else:
    sdp_config = SDPSolverConfig(backend="internal", tol=1e-8)
sos_demo = sos_decompose(sos_polynomial, [x, y], solver_config=sdp_config)
assert sos_demo.certificate is not None
assert sos_demo.certificate.verified
assert sp.expand(sos_demo.exact_residual) == 0
print(sos_demo.sdp_status, sos_demo.certificate.pretty())


## 10. Critical points, KKT and critical cones

In [ ]:
critical_cone_demo = analyze_critical_cone(
    x**2 - y**2, x**2 - y**2, [x, y], {x: 0, y: 0}, [], [], []
)
assert critical_cone_demo.classification == "IndefiniteCriticalCone"
critical_cone_demo


## 11. Transcendental + verified interval optimization

In [ ]:
interval_demo = enclose_expression(sp.sin(x) + x**2, (x,), ((-1.0, 1.0),))
assert interval_demo.rigorous
interval_demo


## 12. Algebraization and structural reductions

In [ ]:
structural_demo = minimize_result(sp.exp((x - 1)**2), variables=x)
assert structural_demo.optimum_value == 1
assert structural_demo.certified
structural_demo.explain()


## 13. Sparsity, separability and moments

In [ ]:
separable_demo = minimize_result((x - 1)**2 + (y + 2)**2, variables=[x, y])
assert separable_demo.optimum_value == 0
sparsity_demo = build_sparsity_graph([x**2 + y**2], [x, y])
moment_demo = build_moment_matrix([x, y], order=1)
assert moment_demo.matrix.shape == (3, 3)
(separable_demo.explain(), sparsity_demo, moment_demo.matrix)


## 14. Diagnostics, explain(), solver_plan

In [ ]:
diagnostic_demo = {
    "method": detailed_result.method,
    "status": detailed_result.status,
    "certified": detailed_result.certified,
    "attained": detailed_result.attained,
    "explanation": detailed_result.explain(),
    "plan": detailed_result.solver_plan.explain() if detailed_result.solver_plan else None,
}
diagnostic_demo


## 15. What symbopt cannot currently solve

General mixed-integer optimization, general parameter-stratified optimizer regions, arbitrary multivariate transcendental theorem proving, and worst-case large dense SOS/KKT systems remain outside the current practical capability set. Planner deferral is a cost decision, not a proof that a route is mathematically impossible. See the limitations page for the maintained list.

## References

See the README and Sphinx manual for API details, solver-selection policy, certification semantics, and the complete references.